In [1]:
# Install required packages
!pip install -q transformers datasets torch accelerate sentencepiece

In [2]:
import json
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import T5Tokenizer, T5ForConditionalGeneration, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch import nn
from tqdm.auto import tqdm
import os
import re
from typing import List, Dict, Tuple
import warnings
import shutil
warnings.filterwarnings('ignore')

2026-02-07 18:41:40.819224: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770489701.005192      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770489701.057916      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770489701.518782      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770489701.518828      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770489701.518831      55 computation_placer.cc:177] computation placer alr

In [3]:
# Configuration with IMPROVED HYPERPARAMETERS
class Config:
    # Paths
    TRAIN_ZIP = '/kaggle/input/dimaste/subtask2'
    DATA_DIR = './subtask2_data'
    OUTPUT_DIR = './subtask2_outputs'
    CHECKPOINT_DIR = './subtask2_checkpoints'
    
    # Model
    MODEL_NAME = 't5-base'
    MAX_INPUT_LENGTH = 128
    MAX_OUTPUT_LENGTH = 256
    
    # Training (IMPROVED!)
    BATCH_SIZE = 16
    GRADIENT_ACCUMULATION_STEPS = 2  # Effective batch = 32
    EPOCHS = 10  # Reduced from 15
    LEARNING_RATE = 5e-5  # Reduced from 3e-4
    WARMUP_RATIO = 0.15  # Increased from 0.1
    WEIGHT_DECAY = 0.01
    GRADIENT_CLIP = 1.0
    LABEL_SMOOTHING = 0.1  # NEW!
    EARLY_STOP_PATIENCE = 3  # NEW!
    
    # VA normalization
    VA_MIN = 1.0
    VA_MAX = 9.0
    
    # Decoding
    NUM_BEAMS = 4
    
    # Device
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # Random seed
    SEED = 42

config = Config()

# Create directories
os.makedirs(config.DATA_DIR, exist_ok=True)
os.makedirs(config.TRAIN_ZIP, exist_ok=True)
os.makedirs(config.OUTPUT_DIR, exist_ok=True)
os.makedirs(config.CHECKPOINT_DIR, exist_ok=True)

# Copy data files
files_to_copy = [
    "eng_laptop_dev_task2.jsonl",
    "eng_laptop_train_alltasks.jsonl",
    "eng_laptop_test_task2.jsonl",
    "eng_restaurant_dev_task2.jsonl",
    "eng_restaurant_test_task2.jsonl",
    "eng_restaurant_train_alltasks.jsonl"
]

for fname in files_to_copy:
    src = os.path.join(config.TRAIN_ZIP, fname)
    dst = os.path.join(config.DATA_DIR, fname)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Copied {fname}")

# Set random seeds
torch.manual_seed(config.SEED)
np.random.seed(config.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config.SEED)

print(f"\nDevice: {config.DEVICE}")
print(f"Effective batch size: {config.BATCH_SIZE * config.GRADIENT_ACCUMULATION_STEPS}")

Copied eng_laptop_dev_task2.jsonl
Copied eng_laptop_train_alltasks.jsonl
Copied eng_laptop_test_task2.jsonl
Copied eng_restaurant_dev_task2.jsonl
Copied eng_restaurant_test_task2.jsonl
Copied eng_restaurant_train_alltasks.jsonl

Device: cuda
Effective batch size: 32


In [4]:
# Data loading functions - CRITICAL FIXES APPLIED

def load_jsonl(file_path):
    """
    Load JSONL data with CRITICAL FIXES:
    1. Convert Quadruplet → Triplet (training data uses Quadruplet)
    2. Filter NULL aspects/opinions (test set has 0% NULLs)
    """
    data = []
    null_filtered = 0
    quadruplet_converted = 0
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line.strip())
            
            # CRITICAL FIX #1: Convert Quadruplet to Triplet
            if 'Quadruplet' in item:
                quadruplet_converted += 1
                item['Triplet'] = []
                
                for quad in item['Quadruplet']:
                    # CRITICAL FIX #2: Filter NULLs
                    if quad['Aspect'] != 'NULL' and quad['Opinion'] != 'NULL':
                        item['Triplet'].append({
                            'Aspect': quad['Aspect'],
                            'Opinion': quad['Opinion'],
                            'VA': quad['VA']
                        })
                    else:
                        null_filtered += 1
                
                del item['Quadruplet']
            
            data.append(item)
    
    # Diagnostic output
    total_triplets = sum(len(item.get('Triplet', [])) for item in data)
    print(f"Loaded {len(data)} samples from {os.path.basename(file_path)}")
    if quadruplet_converted > 0:
        print(f"  ✓ Converted {quadruplet_converted} Quadruplets to Triplets")
        print(f"  ✓ Filtered {null_filtered} NULL aspects/opinions")
    print(f"  ✓ Total triplets: {total_triplets}")
    
    return data

def parse_va(va_string):
    """Parse VA string 'V.VV#A.AA' to (valence, arousal)"""
    v, a = va_string.split('#')
    return float(v), float(a)

def format_va(valence, arousal):
    """Format VA to 'V.VV#A.AA' with clipping"""
    v = np.clip(valence, config.VA_MIN, config.VA_MAX)
    a = np.clip(arousal, config.VA_MIN, config.VA_MAX)
    return f"{v:.2f}#{a:.2f}"

In [5]:
# Format conversion for T5 seq2seq

def triplets_to_sequence(triplets):
    """
    Convert triplet list to sequence.
    Format: [A]aspect[O]opinion[V]VA[SEP]...
    """
    if not triplets:
        return "none"
    
    parts = []
    for triplet in triplets:
        aspect = triplet['Aspect']
        opinion = triplet['Opinion']
        va = triplet['VA']
        parts.append(f"[A]{aspect}[O]{opinion}[V]{va}")
    
    return "[SEP]".join(parts)

def sequence_to_triplets(sequence):
    """
    Parse T5 output back to triplets.
    IMPROVED: Better error handling
    """
    if sequence.strip().lower() in ["none", ""]:
        return []
    
    triplets = []
    pattern = r'\[A\](.*?)\[O\](.*?)\[V\](.*?)(?:\[SEP\]|$)'
    matches = re.finditer(pattern, sequence)
    
    for match in matches:
        aspect = match.group(1).strip()
        opinion = match.group(2).strip()
        va = match.group(3).strip()
        
        # Skip empty or NULL
        if not aspect or not opinion or aspect == 'NULL' or opinion == 'NULL':
            continue
        
        # Validate VA
        if '#' in va:
            try:
                v, a = parse_va(va)
                va = format_va(v, a)
                triplets.append({
                    'Aspect': aspect,
                    'Opinion': opinion,
                    'VA': va
                })
            except:
                continue
        else:
            # Try to salvage malformed VA
            try:
                val = float(va)
                va = format_va(val, val)
                triplets.append({
                    'Aspect': aspect,
                    'Opinion': opinion,
                    'VA': va
                })
            except:
                continue
    
    return triplets

In [6]:
# Dataset classes

class DimASTEDataset(Dataset):
    def __init__(self, data, tokenizer, max_input_length, max_output_length):
        self.data = data
        self.tokenizer = tokenizer
        self.max_input_length = max_input_length
        self.max_output_length = max_output_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        text = item['Text']
        triplets = item.get('Triplet', [])  # Now works for both formats!
        
        input_text = f"extract triplets: {text}"
        target_text = triplets_to_sequence(triplets)
        
        # Tokenize
        input_encoding = self.tokenizer(
            input_text,
            max_length=self.max_input_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        target_encoding = self.tokenizer(
            target_text,
            max_length=self.max_output_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        labels = target_encoding['input_ids'].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100
        
        return {
            'input_ids': input_encoding['input_ids'].squeeze(),
            'attention_mask': input_encoding['attention_mask'].squeeze(),
            'labels': labels
        }

class DimASTETestDataset(Dataset):
    def __init__(self, data, tokenizer, max_input_length):
        self.data = data
        self.tokenizer = tokenizer
        self.max_input_length = max_input_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        text = item['Text']
        input_text = f"extract triplets: {text}"
        
        input_encoding = self.tokenizer(
            input_text,
            max_length=self.max_input_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': input_encoding['input_ids'].squeeze(),
            'attention_mask': input_encoding['attention_mask'].squeeze()
        }

In [7]:
# Evaluation metrics - NEW!

def calculate_triplet_f1(predictions, targets):
    """
    Calculate F1 for triplet extraction.
    Matches on (Aspect, Opinion) pairs.
    """
    pred_set = set()
    target_set = set()
    
    for pred_triplets, target_triplets in zip(predictions, targets):
        for t in pred_triplets:
            pred_set.add((t['Aspect'], t['Opinion']))
        for t in target_triplets:
            target_set.add((t['Aspect'], t['Opinion']))
    
    if len(pred_set) == 0 and len(target_set) == 0:
        return 1.0, 1.0, 1.0
    if len(pred_set) == 0:
        return 0.0, 0.0, 0.0
    
    tp = len(pred_set & target_set)
    precision = tp / len(pred_set) if len(pred_set) > 0 else 0
    recall = tp / len(target_set) if len(target_set) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return precision, recall, f1

def evaluate_va_quality(predictions, targets):
    """
    Calculate RMSE for VA predictions.
    Only for correctly extracted triplets.
    """
    all_pred_v, all_pred_a = [], []
    all_tgt_v, all_tgt_a = [], []
    
    for pred_triplets, target_triplets in zip(predictions, targets):
        for pt in pred_triplets:
            for tt in target_triplets:
                if pt['Aspect'] == tt['Aspect'] and pt['Opinion'] == tt['Opinion']:
                    try:
                        pv, pa = parse_va(pt['VA'])
                        tv, ta = parse_va(tt['VA'])
                        all_pred_v.append(pv)
                        all_pred_a.append(pa)
                        all_tgt_v.append(tv)
                        all_tgt_a.append(ta)
                    except:
                        continue
    
    if len(all_pred_v) == 0:
        return None, None, 0.0
    
    rmse_v = np.sqrt(np.mean((np.array(all_pred_v) - np.array(all_tgt_v))**2))
    rmse_a = np.sqrt(np.mean((np.array(all_pred_a) - np.array(all_tgt_a))**2))
    variance = np.var(all_pred_v + all_pred_a)
    
    return rmse_v, rmse_a, variance

In [8]:
# Training functions with gradient accumulation

def train_epoch(model, dataloader, optimizer, scheduler, device, accumulation_steps=1):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    
    progress_bar = tqdm(dataloader, desc="Training")
    
    for i, batch in enumerate(progress_bar):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        loss = outputs.loss / accumulation_steps
        loss.backward()
        
        if (i + 1) % accumulation_steps == 0:
            nn.utils.clip_grad_norm_(model.parameters(), config.GRADIENT_CLIP)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        
        total_loss += loss.item() * accumulation_steps
        progress_bar.set_postfix({'loss': total_loss / (i + 1)})
    
    # Handle remaining gradients
    if len(dataloader) % accumulation_steps != 0:
        nn.utils.clip_grad_norm_(model.parameters(), config.GRADIENT_CLIP)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
    
    return total_loss / len(dataloader)

def evaluate(model, dataloader, tokenizer, device):
    """
    Evaluation with comprehensive metrics.
    """
    model.eval()
    total_loss = 0
    all_predictions = []
    all_targets = []
    
    progress_bar = tqdm(dataloader, desc="Evaluating")
    
    with torch.no_grad():
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            # Loss
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            total_loss += outputs.loss.item()
            
            # Generate
            generated_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_length=config.MAX_OUTPUT_LENGTH,
                num_beams=config.NUM_BEAMS,
                early_stopping=True
            )
            
            # Decode
            generated_texts = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
            labels_for_decode = labels.clone()
            labels_for_decode[labels_for_decode == -100] = tokenizer.pad_token_id
            target_texts = tokenizer.batch_decode(labels_for_decode, skip_special_tokens=True)
            
            # Parse
            for gen, tgt in zip(generated_texts, target_texts):
                all_predictions.append(sequence_to_triplets(gen))
                all_targets.append(sequence_to_triplets(tgt))
    
    avg_loss = total_loss / len(dataloader)
    precision, recall, f1 = calculate_triplet_f1(all_predictions, all_targets)
    rmse_v, rmse_a, va_var = evaluate_va_quality(all_predictions, all_targets)
    
    return avg_loss, precision, recall, f1, rmse_v, rmse_a, va_var

In [9]:
# Prediction functions

def predict(model, dataloader, tokenizer, device):
    model.eval()
    all_predictions = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Predicting"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            
            generated_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_length=config.MAX_OUTPUT_LENGTH,
                num_beams=config.NUM_BEAMS,
                early_stopping=True
            )
            
            generated_texts = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
            
            for text in generated_texts:
                all_predictions.append(sequence_to_triplets(text))
    
    return all_predictions

def save_predictions(predictions, original_data, output_path):
    with open(output_path, 'w', encoding='utf-8') as f:
        for pred, item in zip(predictions, original_data):
            output_item = {
                'ID': item['ID'],
                'Text': item['Text'],
                'Triplet': pred
            }
            f.write(json.dumps(output_item, ensure_ascii=False) + '\n')
    print(f"Saved: {output_path}")

In [10]:
# Load data - CRITICAL FIXES APPLIED
print("="*80)
print("LOADING DATA")
print("="*80)

# Load training data (auto-converts Quadruplet→Triplet and filters NULLs)
train_data = load_jsonl(f'{config.DATA_DIR}/eng_laptop_train_alltasks.jsonl')

# Load dev data
dev_data = load_jsonl(f'{config.DATA_DIR}/eng_laptop_dev_task2.jsonl')

# VERIFICATION: Check training targets
print("\n🔍 VERIFICATION:")
sample_target = triplets_to_sequence(train_data[0].get('Triplet', []))
print(f"Sample training target: {sample_target}")
if sample_target == "none":
    print("⚠️  WARNING: Data loading failed!")
else:
    print("✅ Training data properly loaded")

LOADING DATA
Loaded 4076 samples from eng_laptop_train_alltasks.jsonl
  ✓ Converted 4076 Quadruplets to Triplets
  ✓ Filtered 2496 NULL aspects/opinions
  ✓ Total triplets: 3277
Loaded 200 samples from eng_laptop_dev_task2.jsonl
  ✓ Total triplets: 317

🔍 VERIFICATION:
Sample training target: [A]unit[O]pretty[V]7.12#7.12[SEP][A]unit[O]stylish[V]7.12#7.12
✅ Training data properly loaded


In [11]:
# Create datasets and dataloaders
print("="*80)
print("CREATING DATASETS")
print("="*80)

tokenizer = T5Tokenizer.from_pretrained(config.MODEL_NAME)

train_dataset = DimASTEDataset(
    train_data, tokenizer,
    config.MAX_INPUT_LENGTH,
    config.MAX_OUTPUT_LENGTH
)

dev_dataset = DimASTEDataset(
    dev_data, tokenizer,
    config.MAX_INPUT_LENGTH,
    config.MAX_OUTPUT_LENGTH
)

train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=config.BATCH_SIZE)

print(f"✓ Training batches: {len(train_loader)}")
print(f"✓ Dev batches: {len(dev_loader)}")

CREATING DATASETS


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


✓ Training batches: 255
✓ Dev batches: 13


In [12]:
# Initialize model
print("="*80)
print("INITIALIZING MODEL")
print("="*80)

model = T5ForConditionalGeneration.from_pretrained(config.MODEL_NAME).to(config.DEVICE)

print(f"✓ Model: {config.MODEL_NAME}")
print(f"✓ Label smoothing: {config.LABEL_SMOOTHING}")
print(f"✓ Parameters: {sum(p.numel() for p in model.parameters()):,}")

INITIALIZING MODEL


model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✓ Model: t5-base
✓ Label smoothing: 0.1
✓ Parameters: 222,903,552


In [13]:
# Setup optimizer and scheduler
print("="*80)
print("SETUP OPTIMIZER")
print("="*80)

optimizer = AdamW(
    model.parameters(),
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY
)

total_steps = len(train_loader) * config.EPOCHS // config.GRADIENT_ACCUMULATION_STEPS
warmup_steps = int(total_steps * config.WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f"✓ Learning rate: {config.LEARNING_RATE}")
print(f"✓ Total steps: {total_steps}")
print(f"✓ Warmup steps: {warmup_steps}")
print(f"✓ Effective batch: {config.BATCH_SIZE * config.GRADIENT_ACCUMULATION_STEPS}")

SETUP OPTIMIZER
✓ Learning rate: 5e-05
✓ Total steps: 1275
✓ Warmup steps: 191
✓ Effective batch: 32


In [14]:
# Training loop with EARLY STOPPING
print("="*80)
print("TRAINING")
print("="*80)

best_loss = float('inf')
best_f1 = 0.0
patience_counter = 0
history = {
    'train_loss': [],
    'dev_loss': [],
    'dev_f1': [],
    'dev_rmse_v': [],
    'dev_rmse_a': []
}

for epoch in range(config.EPOCHS):
    print(f"\n{'='*80}")
    print(f"Epoch {epoch + 1}/{config.EPOCHS}")
    print('='*80)
    
    # Train
    train_loss = train_epoch(
        model, train_loader, optimizer, scheduler,
        config.DEVICE, config.GRADIENT_ACCUMULATION_STEPS
    )
    
    # Evaluate
    dev_loss, precision, recall, f1, rmse_v, rmse_a, va_var = evaluate(
        model, dev_loader, tokenizer, config.DEVICE
    )
    
    # Record
    history['train_loss'].append(train_loss)
    history['dev_loss'].append(dev_loss)
    history['dev_f1'].append(f1)
    if rmse_v is not None:
        history['dev_rmse_v'].append(rmse_v)
        history['dev_rmse_a'].append(rmse_a)
    
    # Print
    print(f"\n📊 Results:")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Dev Loss:   {dev_loss:.4f}")
    print(f"  Precision:  {precision:.4f}")
    print(f"  Recall:     {recall:.4f}")
    print(f"  F1 Score:   {f1:.4f}")
    
    if rmse_v is not None:
        print(f"  VA RMSE (V): {rmse_v:.4f}")
        print(f"  VA RMSE (A): {rmse_a:.4f}")
        print(f"  VA Variance: {va_var:.4f}")
        if va_var < 0.5:
            print(f"  ⚠️  LOW VA VARIANCE - possible collapse!")
    
    # Check improvement
    improved = False
    if dev_loss < best_loss:
        best_loss = dev_loss
        improved = True
    if f1 > best_f1:
        best_f1 = f1
        improved = True
    
    # Save best
    if improved:
        patience_counter = 0
        model.save_pretrained(f'{config.CHECKPOINT_DIR}/best_model')
        tokenizer.save_pretrained(f'{config.CHECKPOINT_DIR}/best_model')
        print(f"  ✅ Saved best model (Loss: {dev_loss:.4f}, F1: {f1:.4f})")
    else:
        patience_counter += 1
        print(f"  No improvement for {patience_counter} epoch(s)")
    
    # Early stopping
    if patience_counter >= config.EARLY_STOP_PATIENCE:
        print(f"\n⚠️  Early stopping at epoch {epoch + 1}")
        print(f"  Best loss: {best_loss:.4f}")
        print(f"  Best F1: {best_f1:.4f}")
        break

print("\n" + "="*80)
print("TRAINING COMPLETED")
print("="*80)
print(f"Best dev loss: {best_loss:.4f}")
print(f"Best F1 score: {best_f1:.4f}")

TRAINING

Epoch 1/10


Training:   0%|          | 0/255 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/13 [00:00<?, ?it/s]


📊 Results:
  Train Loss: 3.3055
  Dev Loss:   0.9239
  Precision:  1.0000
  Recall:     0.0068
  F1 Score:   0.0135
  VA RMSE (V): 0.3636
  VA RMSE (A): 0.6374
  VA Variance: 0.0625
  ⚠️  LOW VA VARIANCE - possible collapse!
  ✅ Saved best model (Loss: 0.9239, F1: 0.0135)

Epoch 2/10


Training:   0%|          | 0/255 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/13 [00:00<?, ?it/s]


📊 Results:
  Train Loss: 0.3994
  Dev Loss:   0.5806
  Precision:  0.4741
  Recall:     0.3729
  F1 Score:   0.4175
  VA RMSE (V): 0.8294
  VA RMSE (A): 0.8768
  VA Variance: 0.6615
  No improvement for 2 epoch(s)

Epoch 10/10


Training:   0%|          | 0/255 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/13 [00:00<?, ?it/s]


📊 Results:
  Train Loss: 0.3973
  Dev Loss:   0.5809
  Precision:  0.4802
  Recall:     0.3695
  F1 Score:   0.4176
  VA RMSE (V): 0.8260
  VA RMSE (A): 0.8538
  VA Variance: 0.6602
  No improvement for 3 epoch(s)

⚠️  Early stopping at epoch 10
  Best loss: 0.5735
  Best F1: 0.4215

TRAINING COMPLETED
Best dev loss: 0.5735
Best F1 score: 0.4215


In [15]:
# Load best model
print("Loading best model...")
model = T5ForConditionalGeneration.from_pretrained(
    f'{config.CHECKPOINT_DIR}/best_model'
).to(config.DEVICE)
tokenizer = T5Tokenizer.from_pretrained(f'{config.CHECKPOINT_DIR}/best_model')
print("✓ Best model loaded")

Loading best model...
✓ Best model loaded


In [16]:
# Generate test predictions
print("="*80)
print("GENERATING TEST PREDICTIONS")
print("="*80)

# Laptop
print("\n📝 Laptop test:")
test_laptop_data = load_jsonl(f'{config.DATA_DIR}/eng_laptop_test_task2.jsonl')
test_laptop_dataset = DimASTETestDataset(test_laptop_data, tokenizer, config.MAX_INPUT_LENGTH)
test_laptop_loader = DataLoader(test_laptop_dataset, batch_size=config.BATCH_SIZE)
predictions_laptop = predict(model, test_laptop_loader, tokenizer, config.DEVICE)
save_predictions(predictions_laptop, test_laptop_data, f'{config.OUTPUT_DIR}/laptop_test_predictions.jsonl')

# Restaurant
print("\n📝 Restaurant test:")
test_rest_data = load_jsonl(f'{config.DATA_DIR}/eng_restaurant_test_task2.jsonl')
test_rest_dataset = DimASTETestDataset(test_rest_data, tokenizer, config.MAX_INPUT_LENGTH)
test_rest_loader = DataLoader(test_rest_dataset, batch_size=config.BATCH_SIZE)
predictions_rest = predict(model, test_rest_loader, tokenizer, config.DEVICE)
save_predictions(predictions_rest, test_rest_data, f'{config.OUTPUT_DIR}/restaurant_test_predictions.jsonl')

GENERATING TEST PREDICTIONS

📝 Laptop test:
Loaded 1000 samples from eng_laptop_test_task2.jsonl
  ✓ Total triplets: 0


Predicting:   0%|          | 0/63 [00:00<?, ?it/s]

Saved: ./subtask2_outputs/laptop_test_predictions.jsonl

📝 Restaurant test:
Loaded 1000 samples from eng_restaurant_test_task2.jsonl
  ✓ Total triplets: 0


Predicting:   0%|          | 0/63 [00:00<?, ?it/s]

Saved: ./subtask2_outputs/restaurant_test_predictions.jsonl


In [18]:
# Sample predictions
print("="*80)
print("SAMPLE PREDICTIONS")
print("="*80)

print("\n🔍 Restaurant (first 3):")
for i in range(min(3, len(predictions_rest))):
    print(f"\n{i+1}. Text: {test_rest_data[i]['Text']}")
    print(f"   Predictions: {predictions_rest[i]}")

print("\n🔍 Laptop (first 3):")
for i in range(min(3, len(predictions_laptop))):
    print(f"\n{i+1}. Text: {test_laptop_data[i]['Text']}")
    print(f"   Predictions: {predictions_laptop[i]}")

print("\n" + "="*80)
print(" ALL DONE!")
print("="*80)

SAMPLE PREDICTIONS

🔍 Restaurant (first 3):

1. Text: My fiance had stewed chicken and I had the curried chicken- both were excellent
   Predictions: [{'Aspect': 'curried chicken', 'Opinion': 'excellent', 'VA': '7.50#7.50'}]

2. Text: But the service , which is a huge reason to spend your hard-earned $ at a restaurant , was absolutely horrendous
   Predictions: [{'Aspect': 'service', 'Opinion': 'absolutely horrendous', 'VA': '3.25#6.50'}]

3. Text: Service was very nice but slow , especially since there were only 3 tables going the whole time we were there
   Predictions: [{'Aspect': 'service', 'Opinion': 'very nice', 'VA': '7.00#7.00'}, {'Aspect': 'service', 'Opinion': 'slow', 'VA': '7.00#7.00'}]

🔍 Laptop (first 3):

1. Text: Granted , the battery life is fairly low , and it's big for a laptop , but this is a desktop replacement , so that was to be expected
   Predictions: []

2. Text: The fans are loud , especially when you are on Performance Mode ( Stamina Mode is slightly quieter 